In [ ]:
import pandas as pd
import os

INPUT_PATH = r"D:\Orbit_degree_LP\jia\all_3_6\1216\n_72_features.csv"
OUTPUT_PATH = r"D:\Orbit_degree_LP\jia\all_3_6\1216\n_72_features_dedup_M.csv"

df = pd.read_csv(INPUT_PATH)

if 'label' not in df.columns:
    raise ValueError("当前文件中没有找到 'label' 列")

m_cols = [c for c in df.columns if c.startswith('M')]
print(f"总共有 {len(m_cols)} 个以 M 开头的特征列。")

df_m = df[m_cols]
df_m_dedup = df_m.T.drop_duplicates(keep='first').T

m_cols_kept = df_m_dedup.columns.tolist()
print(f"去重后保留了 {len(m_cols_kept)} 个 M 列。")

removed_m_cols = sorted(set(m_cols) - set(m_cols_kept))
print(f"被删除的重复 M 列数：{len(removed_m_cols)}")
if removed_m_cols:
    print("被删除的列示例：", removed_m_cols[:10])

other_cols = [c for c in df.columns if not c.startswith('M')]

df_new = pd.concat([df[other_cols], df_m_dedup], axis=1)

zero_cols = [c for c in df_new.columns if (df_new[c] == 0).all()]

# zero_cols = [c for c in df_new.columns if c != 'label' and (df_new[c] == 0).all()]

print(f"检测到 {len(zero_cols)} 个全为 0 的列。")
if zero_cols:
    print("全 0 列示例：", zero_cols[:10])
    df_new = df_new.drop(columns=zero_cols)

cols = df_new.columns.tolist()
cols = ['label'] + [c for c in cols if c != 'label']
df_new = df_new[cols]

df_new.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"去重 + 删除全 0 列后的数据已保存到：{OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import warnings
import os

warnings.filterwarnings('ignore')

def select_and_save_is_features(file_path, output_file="n_72_features_dedup_M6.csv"):
    print(f"正在读取数据: {file_path} ...")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"错误: 无法读取文件. {e}")
        return

    current_basis_cols = ['M1']

    all_m_cols = [f"M{i}" for i in range(1, 546) if f"M{i}" in df.columns]

    def natural_key(string_):
        import re
        return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]
    
    all_m_cols.sort(key=natural_key)
    
    target_cols = [c for c in all_m_cols if c not in current_basis_cols]
    
    print("\n" + "="*60)
    print("开始动态特征筛选 (Dynamic Feature Selection, 使用 Ridge Regression)")
    print(f"初始基底: {current_basis_cols}")
    print(f"待筛选特征数: {len(target_cols)}")
    print("="*60)

    scaler = StandardScaler()
    kept_list = list(current_basis_cols)
    dropped_list = []

    for target in target_cols:
        y = df[target].values
        y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

        basis_data = df[kept_list].values
        
        # 二阶多项式扩展（含线性 + 平方 + 交互）
        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly = poly.fit_transform(basis_data)
        X_scaled = scaler.fit_transform(X_poly)
        
        model = Ridge(alpha=0.01) 
        model.fit(X_scaled, y_scaled)
        y_pred = model.predict(X_scaled)
        r2 = r2_score(y_scaled, y_pred)
        
        is_redundant = r2 > 0.999  
        
        if is_redundant:
            dropped_list.append(target)
            print(f"[剔除] {target:<5} (R2={r2:.5f}) -> 冗余（可由当前基底近似生成）")
        else:
            kept_list.append(target)
            print(f"[保留] {target:<5} (R2={r2:.5f}) -> 不可约（加入基底，残差非忽略不计）")

    print("\n" + "="*60)
    print("筛选完成")
    print(f"原始特征数: {len(all_m_cols)}")
    print(f"保留特征数: {len(kept_list)}")
    print(f"剔除特征数: {len(dropped_list)}")
    print("-" * 60)
    print(f"最终不可约集合 (ISs): {kept_list}")
    
    out_df = pd.DataFrame(kept_list, columns=['Selected_Features'])
    out_path = os.path.join(os.path.dirname(file_path), output_file)
    out_df.to_csv(out_path, index=False)
    print(f"\n 特征列表已保存至: {out_path}")
    
    return kept_list

if __name__ == "__main__":
    file_path = r"D:\Orbit_degree_LP\jia\all_3_6\1216\n_72_features_dedup_M.csv"
    final_features = select_and_save_is_features(file_path)